# Assignment 5 · Task 2 — Parts B & C: NST & Video Pipeline  [Enhanced]

This notebook runs the **enhanced** pipeline with:
- **NST**: Adaptive layer weights, Total Variation regularisation, histogram warm-start, multi-scale optimisation
- **Video**: Optical-flow-guided temporal warping, alpha feathering, H.264 re-encoding
- **Ablations**: Full grid of every content × every style × every condition, white-background layout


## 1 · Environment Check

In [ ]:
import subprocess, sys, os, shutil, re
from pathlib import Path

print('=' * 60)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
!df -h /kaggle/working

## 2 · Clone GitHub Repo
> **Edit `GITHUB_REPO`** to your actual repository URL before running.

In [ ]:
# ✏️  CHANGE THIS
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/CV-AS-5-TASK2'

REPO_DIR = Path('/kaggle/working/repo')
if REPO_DIR.exists():
    print('Repo present — pulling latest ...')
    !git -C {REPO_DIR} pull
else:
    !git clone --depth 1 {GITHUB_REPO} {REPO_DIR}

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('\n[OK] Repo ready.')

In [ ]:
# ── Override with enhanced implementations ──────────────────────────
import json
from pathlib import Path

for fname, code in [
    ('nst.py', "\"\"\"\nnst.py\n\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nNeural Style Transfer (Gatys, Ecker, Bethge \u2014 CVPR 2015)\nAssignment 5 \u2014 Task 2  Part B  [ENHANCED]\n\nKey upgrades over baseline:\n  \u2022 Adaptive per-layer style weights (emphasise shallow + mid layers more)\n  \u2022 Total Variation (TV) regularisation to reduce pixel noise\n  \u2022 Histogram-matching warm-start: initialise generated image by colour-\n    matching the content to the style palette before optimisation begins.\n    This collapses the effective search space and produces richer blends.\n  \u2022 Multi-scale optimisation (coarse\u2192fine): first optimise at half resolution,\n    then upsample and refine at full resolution.  Much better fine detail.\n  \u2022 L-BFGS max_iter tuned per scale; Adam fallback with cosine LR schedule.\n  \u2022 Clamping inside closure using in-place detach trick (avoids autograd graph\n    corruption with L-BFGS).\n\nArchitecture\n  \u2022 Backbone  : pretrained VGG19, FROZEN, eval() mode \u2014 never updated.\n  \u2022 Content   : relu4_2  (mid-level structural features)\n  \u2022 Style     : relu1_1, relu2_1, relu3_1, relu4_1, relu5_1\n                \u2192 Gram matrix of feature activations, normalised by H\u00d7W\u00d7C.\n  \u2022 Optimise  : pixels of the generated image (initialised from content frame).\n  \u2022 Loss      : L_total = \u03b1 \u00d7 L_content + \u03b2 \u00d7 L_style + \u03b3 \u00d7 L_TV\n  \u2022 Sweep     : three \u03b2/\u03b1 ratios \u2014 1e3, 1e5, 1e7.\n\nTemporal consistency (Part C integration)\n  \u2022 When `init_tensor` is supplied the optimisation starts from that tensor\n    instead of the content frame, dramatically reducing inter-frame flicker.\n\nUsage (standalone):\n    python nst.py --content content.jpg --style style.jpg --out out.png\n    python nst.py --content content.jpg --style style.jpg \\\\\n                  --beta_ratio 1e5 --steps 300 --optim lbfgs --multiscale\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport sys\nfrom pathlib import Path\nfrom typing import Optional\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom torchvision import models, transforms\nfrom torchvision.models import VGG19_Weights\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  VGG19 layer-name \u2192 module index mapping (features sequential)\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nVGG19_LAYER_MAP: dict[str, int] = {\n    \"conv1_1\": 0,  \"relu1_1\": 1,\n    \"conv1_2\": 2,  \"relu1_2\": 3,\n    \"pool1\"  : 4,\n    \"conv2_1\": 5,  \"relu2_1\": 6,\n    \"conv2_2\": 7,  \"relu2_2\": 8,\n    \"pool2\"  : 9,\n    \"conv3_1\": 10, \"relu3_1\": 11,\n    \"conv3_2\": 12, \"relu3_2\": 13,\n    \"conv3_3\": 14, \"relu3_3\": 15,\n    \"conv3_4\": 16, \"relu3_4\": 17,\n    \"pool3\"  : 18,\n    \"conv4_1\": 19, \"relu4_1\": 20,\n    \"conv4_2\": 21, \"relu4_2\": 22,\n    \"conv4_3\": 23, \"relu4_3\": 24,\n    \"conv4_4\": 25, \"relu4_4\": 26,\n    \"pool4\"  : 27,\n    \"conv5_1\": 28, \"relu5_1\": 29,\n    \"conv5_2\": 30, \"relu5_2\": 31,\n    \"conv5_3\": 32, \"relu5_3\": 33,\n    \"conv5_4\": 34, \"relu5_4\": 35,\n    \"pool5\"  : 36,\n}\n\n# Normalisation constants for VGG19 (ImageNet mean / std)\n_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])\n_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])\n\n# Default adaptive layer weights: shallow layers carry more style signal\n# (textures), deeper layers add structure.  Weights are then normalised inside\n# the style loss so the absolute magnitudes remain stable as layers are added/\n# removed.\n_DEFAULT_STYLE_LAYER_WEIGHTS = {\n    \"relu1_1\": 1.0,   # fine textures (highest weight)\n    \"relu2_1\": 0.8,\n    \"relu3_1\": 0.6,\n    \"relu4_1\": 0.4,\n    \"relu5_1\": 0.2,   # coarse structure (lowest weight)\n}\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Feature extractor\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\nclass VGG19FeatureExtractor(nn.Module):\n    \"\"\"\n    Pretrained VGG19 feature extractor.\n\n    \u2022 Always in eval() mode.\n    \u2022 Parameters are frozen (requires_grad=False).\n    \u2022 Returns a dict of named intermediate activations.\n    \u2022 Applies ImageNet normalisation internally so callers pass [0,1] tensors.\n    \"\"\"\n\n    def __init__(self, layer_names: list[str], device: torch.device):\n        super().__init__()\n\n        vgg = models.vgg19(weights=VGG19_Weights.IMAGENET1K_V1)\n        self.features: nn.Sequential = vgg.features\n\n        # Freeze ALL parameters \u2014 VGG is never updated\n        for param in self.features.parameters():\n            param.requires_grad_(False)\n        self.features.eval()\n\n        # Resolve requested layer names \u2192 feature indices\n        self._layer_names = layer_names\n        self._layer_indices: dict[str, int] = {}\n        for name in layer_names:\n            if name not in VGG19_LAYER_MAP:\n                raise ValueError(\n                    f\"Unknown VGG19 layer: {name!r}. \"\n                    f\"Valid options: {sorted(VGG19_LAYER_MAP.keys())}\"\n                )\n            self._layer_indices[name] = VGG19_LAYER_MAP[name]\n\n        self._max_idx = max(self._layer_indices.values())\n\n        # ImageNet normalisation (registered as buffers \u2014 move with .to(device))\n        self.register_buffer(\"mean\", _IMAGENET_MEAN.view(1, 3, 1, 1))\n        self.register_buffer(\"std\",  _IMAGENET_STD.view(1, 3, 1, 1))\n\n        self.to(device)\n\n    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:\n        \"\"\"\n        Parameters\n        ----------\n        x : (B, 3, H, W) tensor, values in [0, 1]\n\n        Returns\n        -------\n        dict  {layer_name: feature_tensor}\n        \"\"\"\n        # ImageNet normalisation\n        x = (x - self.mean) / self.std\n\n        outputs: dict[str, torch.Tensor] = {}\n        for idx, layer in enumerate(self.features):\n            x = layer(x)\n            for name, target_idx in self._layer_indices.items():\n                if idx == target_idx:\n                    outputs[name] = x\n            if idx >= self._max_idx:\n                break\n\n        return outputs\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Gram matrix\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef gram_matrix(feat: torch.Tensor) -> torch.Tensor:\n    \"\"\"\n    Compute normalised Gram matrix of a feature map.\n\n    Parameters\n    ----------\n    feat : (B, C, H, W)\n\n    Returns\n    -------\n    G    : (B, C, C)  \u2014 G_ij = (1 / C\u00b7H\u00b7W) \u00b7 \u03a3_hw feat_ih \u00b7 feat_jh\n    \"\"\"\n    B, C, H, W = feat.shape\n    f = feat.view(B, C, H * W)\n    G = torch.bmm(f, f.transpose(1, 2))\n    return G / (C * H * W)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Loss functions\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef content_loss(gen_feat: torch.Tensor, content_feat: torch.Tensor) -> torch.Tensor:\n    \"\"\"Mean-squared difference between generated and content feature maps.\"\"\"\n    return F.mse_loss(gen_feat, content_feat)\n\n\ndef style_loss(\n    gen_feats:   dict[str, torch.Tensor],\n    style_grams: dict[str, torch.Tensor],\n    layer_weights: dict[str, float],\n) -> torch.Tensor:\n    \"\"\"\n    Weighted sum of MSE between Gram matrices across style layers.\n    Weights are normalised so they sum to 1 \u2014 this keeps the overall\n    style loss magnitude independent of which/how many layers are used.\n    \"\"\"\n    total_w = sum(layer_weights[n] for n in style_grams)\n    if total_w == 0:\n        total_w = 1.0\n\n    loss = torch.tensor(0.0, device=next(iter(gen_feats.values())).device)\n    for name in style_grams:\n        w       = layer_weights.get(name, 1.0) / total_w\n        G_gen   = gram_matrix(gen_feats[name])\n        G_style = style_grams[name]\n        loss    = loss + w * F.mse_loss(G_gen, G_style)\n    return loss\n\n\ndef total_variation_loss(x: torch.Tensor) -> torch.Tensor:\n    \"\"\"\n    Anisotropic Total Variation regularisation.\n    Encourages spatial smoothness in the generated image.\n    \"\"\"\n    tv_h = torch.mean(torch.abs(x[:, :, 1:, :] - x[:, :, :-1, :]))\n    tv_w = torch.mean(torch.abs(x[:, :, :, 1:] - x[:, :, :, :-1]))\n    return tv_h + tv_w\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Image helpers\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef load_image(path: Path | str, size: int | None = None) -> torch.Tensor:\n    \"\"\"\n    Load a PIL image \u2192 (1, 3, H, W) float tensor in [0, 1].\n    \"\"\"\n    img = Image.open(path).convert(\"RGB\")\n    if size is not None:\n        w, h = img.size\n        if h < w:\n            new_h = size\n            new_w = int(w * size / h)\n        else:\n            new_w = size\n            new_h = int(h * size / w)\n        img = img.resize((new_w, new_h), Image.LANCZOS)\n    t = transforms.ToTensor()(img)\n    return t.unsqueeze(0)\n\n\ndef tensor_to_pil(t: torch.Tensor) -> Image.Image:\n    \"\"\"Convert (1, 3, H, W) or (3, H, W) tensor in [0,1] \u2192 PIL RGB Image.\"\"\"\n    t = t.squeeze(0).clamp(0, 1).cpu()\n    return transforms.ToPILImage()(t)\n\n\ndef histogram_match(source: torch.Tensor, reference: torch.Tensor) -> torch.Tensor:\n    \"\"\"\n    Match the colour histogram of `source` to `reference`.\n    Both are (1, 3, H, W) tensors in [0, 1].\n    Returns a colour-adjusted version of `source` with matched palette.\n\n    Uses channel-wise CDF matching (Reinhard-style).\n    \"\"\"\n    out = source.clone()\n    for c in range(3):\n        src = source[0, c].cpu().numpy().flatten()\n        ref = reference[0, c].cpu().numpy().flatten()\n\n        # CDF of source\n        src_sorted = np.sort(src)\n        src_counts, src_bins = np.histogram(src, bins=256, range=(0, 1))\n        src_cdf = np.cumsum(src_counts).astype(float) / src_counts.sum()\n        src_bin_centers = (src_bins[:-1] + src_bins[1:]) / 2\n\n        # CDF of reference\n        ref_counts, ref_bins = np.histogram(ref, bins=256, range=(0, 1))\n        ref_cdf = np.cumsum(ref_counts).astype(float) / ref_counts.sum()\n        ref_bin_centers = (ref_bins[:-1] + ref_bins[1:]) / 2\n\n        # Build lookup: for each source bin value, find matching ref bin\n        lookup = np.interp(src_cdf, ref_cdf, ref_bin_centers)\n\n        # Map source pixels\n        src_mapped = np.interp(src, src_bin_centers, lookup)\n        h, w = source.shape[2], source.shape[3]\n        out[0, c] = torch.tensor(src_mapped.reshape(h, w), dtype=torch.float32)\n\n    return out.clamp(0, 1)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Core NST optimisation\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef run_nst(\n    content_tensor: torch.Tensor,\n    style_tensor:   torch.Tensor,\n    device:         torch.device,\n    content_layer:  str  = \"relu4_2\",\n    style_layers:   list[str] = None,\n    style_layer_weights: list[float] = None,\n    content_weight: float = 1.0,        # \u03b1\n    style_weight:   float = 1e5,        # \u03b2\n    tv_weight:      float = 1e-4,       # \u03b3 \u2014 TV regularisation\n    num_steps:      int   = 300,\n    optimizer:      str   = \"lbfgs\",    # \"lbfgs\" | \"adam\"\n    init_tensor:    Optional[torch.Tensor] = None,\n    histogram_init: bool  = True,       # colour-match warm-start\n    multiscale:     bool  = False,      # coarse\u2192fine optimisation\n    verbose:        bool  = True,\n    log_every:      int   = 50,\n) -> torch.Tensor:\n    \"\"\"\n    Run Neural Style Transfer.\n\n    Parameters\n    ----------\n    content_tensor       : (1, 3, H, W) in [0,1]  \u2014 the content image / frame\n    style_tensor         : (1, 3, H, W) in [0,1]  \u2014 the style artwork\n    device               : torch.device\n    content_layer        : VGG19 layer name for content loss\n    style_layers         : list of VGG19 layer names for style loss\n    style_layer_weights  : per-layer weights (defaults to adaptive schedule)\n    content_weight       : \u03b1 \u2014 scaling factor for content loss\n    style_weight         : \u03b2 \u2014 scaling factor for style loss\n    tv_weight            : \u03b3 \u2014 total variation regularisation weight\n    num_steps            : number of optimisation steps\n    optimizer            : \"lbfgs\" or \"adam\"\n    init_tensor          : optional (1,3,H,W) tensor to initialise from\n                           (supply previous stylised frame for temporal consistency)\n    histogram_init       : apply histogram matching warm-start when init_tensor=None\n    multiscale           : run coarse\u2192fine two-stage optimisation\n    verbose              : print loss every log_every steps\n    log_every            : print interval\n\n    Returns\n    -------\n    (1, 3, H, W) float tensor in [0, 1] \u2014 stylised image\n    \"\"\"\n    if style_layers is None:\n        style_layers = [\"relu1_1\", \"relu2_1\", \"relu3_1\", \"relu4_1\", \"relu5_1\"]\n\n    # Build adaptive layer weight map\n    if style_layer_weights is None:\n        layer_weight_map = {\n            name: _DEFAULT_STYLE_LAYER_WEIGHTS.get(name, 1.0)\n            for name in style_layers\n        }\n    else:\n        layer_weight_map = dict(zip(style_layers, style_layer_weights))\n\n    # Multi-scale: first run at half-res, upsample, then refine at full-res\n    if multiscale:\n        H, W = content_tensor.shape[2], content_tensor.shape[3]\n        H_half, W_half = max(H // 2, 64), max(W // 2, 64)\n\n        # Coarse stage (half resolution, fewer steps)\n        c_half = F.interpolate(content_tensor, (H_half, W_half), mode=\"bilinear\", align_corners=False)\n        s_half = F.interpolate(style_tensor,   (H_half, W_half), mode=\"bilinear\", align_corners=False)\n        init_half = None\n        if init_tensor is not None:\n            init_half = F.interpolate(init_tensor, (H_half, W_half), mode=\"bilinear\", align_corners=False)\n\n        coarse = _run_nst_single_scale(\n            c_half, s_half, device,\n            content_layer, style_layers, layer_weight_map,\n            content_weight, style_weight, tv_weight,\n            max(num_steps // 3, 50), optimizer,\n            init_half, histogram_init, verbose=False,\n        )\n\n        # Upsample coarse result to full res for fine-stage init\n        fine_init = F.interpolate(coarse, (H, W), mode=\"bilinear\", align_corners=False)\n\n        # Fine stage (full resolution, remaining steps)\n        return _run_nst_single_scale(\n            content_tensor, style_tensor, device,\n            content_layer, style_layers, layer_weight_map,\n            content_weight, style_weight, tv_weight,\n            num_steps, optimizer,\n            fine_init, histogram_init=False,  # already matched in coarse stage\n            verbose=verbose, log_every=log_every,\n        )\n    else:\n        return _run_nst_single_scale(\n            content_tensor, style_tensor, device,\n            content_layer, style_layers, layer_weight_map,\n            content_weight, style_weight, tv_weight,\n            num_steps, optimizer,\n            init_tensor, histogram_init,\n            verbose=verbose, log_every=log_every,\n        )\n\n\ndef _run_nst_single_scale(\n    content_tensor:  torch.Tensor,\n    style_tensor:    torch.Tensor,\n    device:          torch.device,\n    content_layer:   str,\n    style_layers:    list[str],\n    layer_weight_map: dict[str, float],\n    content_weight:  float,\n    style_weight:    float,\n    tv_weight:       float,\n    num_steps:       int,\n    optimizer:       str,\n    init_tensor:     Optional[torch.Tensor],\n    histogram_init:  bool,\n    verbose:         bool,\n    log_every:       int = 50,\n) -> torch.Tensor:\n    \"\"\"Single-scale NST optimisation loop.\"\"\"\n\n    all_layers = list(set([content_layer] + style_layers))\n    extractor = VGG19FeatureExtractor(all_layers, device)\n    extractor.eval()\n\n    content_tensor = content_tensor.to(device)\n    style_tensor   = style_tensor.to(device)\n\n    # Resize style to match content spatial dims\n    if style_tensor.shape[2:] != content_tensor.shape[2:]:\n        style_tensor = F.interpolate(\n            style_tensor, size=content_tensor.shape[2:],\n            mode=\"bilinear\", align_corners=False,\n        )\n\n    # Pre-compute targets\n    with torch.no_grad():\n        content_feats  = extractor(content_tensor)\n        content_target = content_feats[content_layer].detach()\n\n        style_feats = extractor(style_tensor)\n        style_grams = {\n            name: gram_matrix(style_feats[name]).detach()\n            for name in style_layers\n        }\n\n    # Initialise generated image\n    if init_tensor is not None:\n        init = init_tensor.clone().to(device)\n        if init.shape[2:] != content_tensor.shape[2:]:\n            init = F.interpolate(init, content_tensor.shape[2:], mode=\"bilinear\", align_corners=False)\n        gen = init.clamp(0, 1)\n    elif histogram_init:\n        # Warm-start: match content colours to style palette\n        gen = histogram_match(content_tensor.cpu(), style_tensor.cpu()).to(device)\n    else:\n        gen = content_tensor.clone()\n\n    gen = gen.requires_grad_(True)\n\n    # Optimiser setup\n    if optimizer.lower() == \"lbfgs\":\n        optim = torch.optim.LBFGS([gen], lr=1.0, max_iter=20,\n                                   line_search_fn=\"strong_wolfe\")\n    elif optimizer.lower() == \"adam\":\n        optim = torch.optim.Adam([gen], lr=0.01)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=num_steps, eta_min=1e-4)\n    else:\n        raise ValueError(f\"Unknown optimizer: {optimizer!r}. Use 'lbfgs' or 'adam'.\")\n\n    step   = [0]\n\n    def closure():\n        # Clamp without breaking the autograd graph (detach + re-attach)\n        with torch.no_grad():\n            gen.data.clamp_(0, 1)\n        optim.zero_grad()\n\n        gen_feats = extractor(gen)\n\n        c_loss = content_loss(gen_feats[content_layer], content_target)\n        s_loss = style_loss(gen_feats, style_grams, layer_weight_map)\n        tv_l   = total_variation_loss(gen)\n\n        total = content_weight * c_loss + style_weight * s_loss + tv_weight * tv_l\n        total.backward()\n\n        step[0] += 1\n        if verbose and step[0] % log_every == 0:\n            print(\n                f\"  Step {step[0]:4d}/{num_steps} | \"\n                f\"Total={total.item():.4e} | \"\n                f\"Content={c_loss.item():.4e} | \"\n                f\"Style={s_loss.item():.4e} | \"\n                f\"TV={tv_l.item():.4e}\"\n            )\n        return total\n\n    if optimizer.lower() == \"lbfgs\":\n        lbfgs_steps = max(1, num_steps // 20)\n        for _ in range(lbfgs_steps):\n            optim.step(closure)\n    else:\n        for _ in range(num_steps):\n            optim.step(closure)\n            scheduler.step()\n\n    with torch.no_grad():\n        gen.data.clamp_(0, 1)\n\n    return gen.detach()\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Style-ratio sweep helper\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef style_ratio_sweep(\n    content_tensor: torch.Tensor,\n    style_tensor:   torch.Tensor,\n    device:         torch.device,\n    beta_ratios:    list[float] = (1e3, 1e5, 1e7),\n    content_weight: float = 1.0,\n    **nst_kwargs,\n) -> dict[float, torch.Tensor]:\n    \"\"\"\n    Run NST for each \u03b2/\u03b1 ratio and return a dict {beta_ratio: stylised_tensor}.\n    \"\"\"\n    results: dict[float, torch.Tensor] = {}\n    for ratio in beta_ratios:\n        beta = ratio * content_weight\n        print(f\"\\n{'\u2500'*60}\")\n        print(f\"  \u03b2/\u03b1 = {ratio:.0e}   (\u03b1={content_weight}, \u03b2={beta:.0e})\")\n        print(f\"{'\u2500'*60}\")\n        out = run_nst(\n            content_tensor=content_tensor,\n            style_tensor=style_tensor,\n            device=device,\n            content_weight=content_weight,\n            style_weight=beta,\n            **nst_kwargs,\n        )\n        results[ratio] = out\n    return results\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  CLI \u2014 standalone usage\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef parse_args() -> argparse.Namespace:\n    p = argparse.ArgumentParser(\n        description=\"Neural Style Transfer \u2014 Part B (Assignment 5, Task 2) [Enhanced]\"\n    )\n    p.add_argument(\"--content\",    required=True, help=\"Path to content image\")\n    p.add_argument(\"--style\",      required=True, help=\"Path to style image\")\n    p.add_argument(\"--out\",        default=\"nst_output.png\",\n                   help=\"Output path (single run) or directory prefix (sweep mode)\")\n    p.add_argument(\"--beta_ratio\", type=float, default=1e5,\n                   help=\"\u03b2/\u03b1 ratio  (default: 1e5). Ignored in --sweep mode.\")\n    p.add_argument(\"--sweep\",      action=\"store_true\",\n                   help=\"Run all three \u03b2/\u03b1 ratios: 1e3, 1e5, 1e7\")\n    p.add_argument(\"--steps\",      type=int, default=300,\n                   help=\"Optimisation steps (default: 300)\")\n    p.add_argument(\"--optim\",      default=\"lbfgs\", choices=[\"lbfgs\", \"adam\"],\n                   help=\"Pixel optimizer (default: lbfgs)\")\n    p.add_argument(\"--size\",       type=int, default=512,\n                   help=\"Shorter-edge resize before NST (default: 512)\")\n    p.add_argument(\"--tv_weight\",  type=float, default=1e-4,\n                   help=\"TV regularisation weight (default: 1e-4)\")\n    p.add_argument(\"--multiscale\", action=\"store_true\",\n                   help=\"Enable coarse\u2192fine multi-scale optimisation\")\n    p.add_argument(\"--no_histogram\", action=\"store_true\",\n                   help=\"Disable histogram-matching warm-start\")\n    p.add_argument(\"--content_layer\", default=\"relu4_2\",\n                   help=\"VGG19 content layer (default: relu4_2)\")\n    p.add_argument(\"--style_layers\",  nargs=\"+\",\n                   default=[\"relu1_1\", \"relu2_1\", \"relu3_1\", \"relu4_1\", \"relu5_1\"],\n                   help=\"VGG19 style layers\")\n    p.add_argument(\"--device\",     default=None,\n                   help=\"Force device: cpu | cuda | cuda:0 (auto if omitted)\")\n    p.add_argument(\"--quiet\",      action=\"store_true\",\n                   help=\"Suppress per-step logs\")\n    return p.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n\n    if args.device:\n        device = torch.device(args.device)\n    else:\n        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"Device : {device}\")\n\n    content = load_image(args.content, size=args.size).to(device)\n    style   = load_image(args.style,   size=args.size).to(device)\n    print(f\"Content : {Path(args.content).name}  {tuple(content.shape)}\")\n    print(f\"Style   : {Path(args.style).name}    {tuple(style.shape)}\")\n\n    nst_kwargs = dict(\n        content_layer=args.content_layer,\n        style_layers=args.style_layers,\n        num_steps=args.steps,\n        optimizer=args.optim,\n        tv_weight=args.tv_weight,\n        multiscale=args.multiscale,\n        histogram_init=not args.no_histogram,\n        verbose=not args.quiet,\n    )\n\n    if args.sweep:\n        results = style_ratio_sweep(\n            content, style, device,\n            beta_ratios=[1e3, 1e5, 1e7],\n            **nst_kwargs,\n        )\n        out_dir = Path(args.out)\n        out_dir.mkdir(parents=True, exist_ok=True)\n        for ratio, t in results.items():\n            fname = out_dir / f\"nst_beta_{ratio:.0e}.png\"\n            tensor_to_pil(t).save(fname)\n            print(f\"  Saved: {fname}\")\n\n        _save_sweep_comparison(\n            content, style, results,\n            out_path=out_dir / \"nst_sweep_comparison.png\",\n        )\n    else:\n        out = run_nst(\n            content, style, device,\n            style_weight=args.beta_ratio,\n            **nst_kwargs,\n        )\n        out_path = Path(args.out)\n        out_path.parent.mkdir(parents=True, exist_ok=True)\n        tensor_to_pil(out).save(out_path)\n        print(f\"\\n\u2713  Saved: {out_path}\")\n\n\ndef _save_sweep_comparison(\n    content: torch.Tensor,\n    style:   torch.Tensor,\n    results: dict[float, torch.Tensor],\n    out_path: Path,\n) -> None:\n    \"\"\"Save a side-by-side comparison of content, style, and all three ratios.\"\"\"\n    try:\n        import matplotlib\n        matplotlib.use(\"Agg\")\n        import matplotlib.pyplot as plt\n        import numpy as np\n\n        cols   = [\"Content\", \"Style\"] + [f\"\u03b2/\u03b1 = {r:.0e}\" for r in results]\n        images = [tensor_to_pil(content), tensor_to_pil(style)] + [\n            tensor_to_pil(t) for t in results.values()\n        ]\n        n = len(images)\n        fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))\n        fig.patch.set_facecolor(\"white\")\n        for ax, img, title in zip(axes, images, cols):\n            ax.imshow(np.array(img))\n            ax.set_title(title, fontsize=12, fontweight=\"bold\", color=\"black\")\n            ax.axis(\"off\")\n\n        plt.suptitle(\"NST Style Ratio Sweep (Gatys et al. 2015) \u2014 Enhanced\", fontsize=14, color=\"black\")\n        plt.tight_layout()\n        plt.savefig(out_path, dpi=120, bbox_inches=\"tight\", facecolor=\"white\")\n        plt.close()\n        print(f\"\\n\u2713  Sweep comparison saved: {out_path}\")\n    except ImportError:\n        print(\"  matplotlib not available \u2014 sweep comparison figure skipped.\")\n\n\nif __name__ == \"__main__\":\n    main()\n"),
    ('video_pipeline.py', "\"\"\"\nvideo_pipeline.py\n\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nPart C \u2014 Video Compositing Pipeline  [ENHANCED]\nAssignment 5 \u2014 Task 2\n\nKey upgrades over baseline:\n  \u2022 Optical-flow-guided temporal warping: warp the previous stylised frame\n    to the current frame using dense Farneback optical flow before using it\n    as the NST init.  This ensures the init is spatially aligned, giving\n    dramatically less ghosting than a raw t\u22121 copy.\n  \u2022 Multi-scale NST per frame: coarse\u2192fine gives richer texture in less time.\n  \u2022 Alpha feathering: erode+Gaussian-blur the alpha matte boundary to avoid\n    sharp compositing seams.\n  \u2022 Histogram warm-start disabled on video (uses warp init instead).\n  \u2022 H.264 re-encoding via ffmpeg after writing mp4v raw \u2014 far smaller files\n    and universally playable.\n  \u2022 Progress bar with ETA and per-frame timing.\n\nPipeline:\n  1. Decode input video into frames (via OpenCV).\n  2. For each frame t:\n       a. Run matting model \u2192 alpha matte \u03b1_t  \u2208 [0,1], feathered at edges\n       b. Run NST (multi-scale, optflow-warped init) \u2192 stylised image S_t\n       c. Composite per pixel:\n            background-stylised: O_t = \u03b1_t \u00b7 F_t + (1\u2212\u03b1_t) \u00b7 S_t\n            subject-stylised:    O_t = \u03b1_t \u00b7 S_t + (1\u2212\u03b1_t) \u00b7 F_t\n  3. Re-encode composited frames \u2192 output video at original frame rate.\n\nUsage:\n    python video_pipeline.py \\\\\n        --video      my_video.mp4 \\\\\n        --style      artwork.jpg \\\\\n        --weights    matting_weights.pth \\\\\n        --out        stylised_output.mp4 \\\\\n        --mode       background \\\\\n        --beta_ratio 1e5\n\n    # Sweep all three \u03b2/\u03b1 ratios (saves three output videos):\n    python video_pipeline.py --video my_video.mp4 --style artwork.jpg \\\\\n        --weights matting_weights.pth --out out_dir/ --sweep\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Optional\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom torchvision import transforms\n\nsys.path.insert(0, str(Path(__file__).parent))\nfrom model import build_matting_model\nfrom nst import (\n    VGG19FeatureExtractor,\n    VGG19_LAYER_MAP,\n    gram_matrix,\n    run_nst,\n    load_image as nst_load_image,\n    tensor_to_pil,\n)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Frame I/O helpers\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef load_matting_model(weights_path: str | Path, arch: str, device: torch.device):\n    \"\"\"Load and return the matting model in eval mode.\"\"\"\n    model = build_matting_model(arch)\n    state = torch.load(weights_path, map_location=device, weights_only=True)\n    model.load_state_dict(state)\n    model.to(device).eval()\n    return model\n\n\ndef bgr_frame_to_tensor(frame_bgr: np.ndarray) -> torch.Tensor:\n    \"\"\"Convert an OpenCV BGR frame (H,W,3) uint8 \u2192 (1,3,H,W) float [0,1].\"\"\"\n    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)\n    t   = transforms.ToTensor()(rgb)\n    return t.unsqueeze(0)\n\n\ndef tensor_to_bgr(t: torch.Tensor) -> np.ndarray:\n    \"\"\"Convert (1,3,H,W) or (3,H,W) float [0,1] \u2192 BGR uint8 ndarray.\"\"\"\n    arr_rgb = (t.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)\n    return cv2.cvtColor(arr_rgb, cv2.COLOR_RGB2BGR)\n\n\n@torch.no_grad()\ndef run_matting(\n    model:      torch.nn.Module,\n    frame_bgr:  np.ndarray,\n    input_size: tuple[int, int],\n    device:     torch.device,\n) -> np.ndarray:\n    \"\"\"\n    Run the matting model on a single BGR frame.\n\n    Returns\n    -------\n    alpha_map : (H, W) float32 in [0, 1], at original frame resolution\n    \"\"\"\n    H_orig, W_orig = frame_bgr.shape[:2]\n\n    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)\n    pil = Image.fromarray(rgb)\n    pil_resized = pil.resize((input_size[1], input_size[0]), Image.BILINEAR)\n\n    tensor   = transforms.ToTensor()(pil_resized).unsqueeze(0).to(device)\n    alpha_t  = model(tensor)\n\n    alpha_resized = F.interpolate(\n        alpha_t, size=(H_orig, W_orig),\n        mode=\"bilinear\", align_corners=False,\n    )\n    return alpha_resized.squeeze().cpu().numpy().astype(np.float32)\n\n\ndef feather_alpha(alpha: np.ndarray, blur_ksize: int = 15, erode_px: int = 3) -> np.ndarray:\n    \"\"\"\n    Feather alpha matte at the boundary:\n      1. Erode slightly to pull edge inward.\n      2. Apply Gaussian blur so the transition is soft.\n    This eliminates sharp compositing seams.\n    \"\"\"\n    a_u8 = (alpha * 255).astype(np.uint8)\n    kernel = np.ones((erode_px, erode_px), dtype=np.uint8)\n    a_eroded = cv2.erode(a_u8, kernel, iterations=1)\n    ksize = blur_ksize if blur_ksize % 2 == 1 else blur_ksize + 1\n    a_blurred = cv2.GaussianBlur(a_eroded, (ksize, ksize), 0)\n    return (a_blurred.astype(np.float32) / 255.0).clip(0, 1)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Optical-flow-guided temporal warping\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef warp_tensor_with_flow(\n    prev_tensor: torch.Tensor,\n    prev_bgr:    np.ndarray,\n    curr_bgr:    np.ndarray,\n) -> torch.Tensor:\n    \"\"\"\n    Warp `prev_tensor` (1,3,H,W) from frame t\u22121 to frame t using dense\n    Farneback optical flow estimated between `prev_bgr` and `curr_bgr`.\n\n    The warp ensures the temporal init is spatially aligned with the current\n    content, reducing ghosting artefacts.\n\n    Returns\n    -------\n    warped : (1, 3, H, W) float tensor in [0, 1], same size as prev_tensor\n    \"\"\"\n    H, W = prev_bgr.shape[:2]\n    prev_gray = cv2.cvtColor(prev_bgr, cv2.COLOR_BGR2GRAY)\n    curr_gray = cv2.cvtColor(curr_bgr, cv2.COLOR_BGR2GRAY)\n\n    # Resize to a fast flow-estimation size\n    flow_h, flow_w = min(H, 256), min(W, 256)\n    pg = cv2.resize(prev_gray, (flow_w, flow_h))\n    cg = cv2.resize(curr_gray, (flow_w, flow_h))\n\n    flow = cv2.calcOpticalFlowFarneback(\n        pg, cg, None,\n        pyr_scale=0.5, levels=3, winsize=15,\n        iterations=3, poly_n=5, poly_sigma=1.2, flags=0,\n    )\n\n    # Scale flow back to the NST tensor size\n    tH, tW = prev_tensor.shape[2], prev_tensor.shape[3]\n    flow_scaled = cv2.resize(flow, (tW, tH))\n    flow_scaled[:, :, 0] *= tW / flow_w\n    flow_scaled[:, :, 1] *= tH / flow_h\n\n    # Build sampling grid\n    grid_y, grid_x = np.mgrid[0:tH, 0:tW].astype(np.float32)\n    map_x = (grid_x + flow_scaled[:, :, 0]).clip(0, tW - 1)\n    map_y = (grid_y + flow_scaled[:, :, 1]).clip(0, tH - 1)\n\n    # Normalise to [-1, 1] for F.grid_sample\n    grid_norm_x = (map_x / (tW - 1)) * 2 - 1\n    grid_norm_y = (map_y / (tH - 1)) * 2 - 1\n    grid = torch.from_numpy(\n        np.stack([grid_norm_x, grid_norm_y], axis=-1)\n    ).unsqueeze(0).float().to(prev_tensor.device)\n\n    warped = F.grid_sample(prev_tensor, grid, mode=\"bilinear\",\n                           padding_mode=\"border\", align_corners=True)\n    return warped.clamp(0, 1)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Compositing\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef composite_frame(\n    frame_bgr:   np.ndarray,\n    stylised_t:  torch.Tensor,\n    alpha_map:   np.ndarray,\n    mode:        str = \"background\",\n) -> np.ndarray:\n    \"\"\"\n    Per-pixel composite with feathered alpha.\n\n    Modes\n    -----\n    \"background\" : O = \u03b1\u00b7F + (1\u2212\u03b1)\u00b7S  \u2014 keep subject, stylise background\n    \"subject\"    : O = \u03b1\u00b7S + (1\u2212\u03b1)\u00b7F  \u2014 stylise subject, keep background\n\n    Parameters\n    ----------\n    frame_bgr   : (H, W, 3) uint8 BGR\n    stylised_t  : (1, 3, H, W) float [0,1] (RGB)\n    alpha_map   : (H, W) float32 [0,1]\n    mode        : \"background\" | \"subject\"\n    \"\"\"\n    H, W = frame_bgr.shape[:2]\n\n    s_np = tensor_to_bgr(\n        F.interpolate(stylised_t, size=(H, W), mode=\"bilinear\", align_corners=False)\n    ).astype(np.float32) / 255.0\n\n    f_np = frame_bgr.astype(np.float32) / 255.0\n    a    = feather_alpha(alpha_map)[:, :, np.newaxis]\n\n    if mode == \"background\":\n        out = a * f_np + (1.0 - a) * s_np\n    elif mode == \"subject\":\n        out = a * s_np + (1.0 - a) * f_np\n    else:\n        raise ValueError(f\"Unknown composite mode: {mode!r}.\")\n\n    return (out.clip(0, 1) * 255).astype(np.uint8)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  ffmpeg re-encode helper\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef reencode_h264(input_path: Path, output_path: Path, crf: int = 18) -> None:\n    \"\"\"\n    Re-encode a raw mp4v video to H.264 using ffmpeg.\n    Produces a much smaller, universally playable file.\n    Falls back silently if ffmpeg is not available.\n    \"\"\"\n    if not shutil.which(\"ffmpeg\"):\n        print(\"  [info] ffmpeg not found \u2014 skipping H.264 re-encode\")\n        if input_path != output_path:\n            shutil.move(str(input_path), str(output_path))\n        return\n\n    tmp = output_path.with_suffix(\".tmp.mp4\")\n    cmd = [\n        \"ffmpeg\", \"-y\",\n        \"-i\", str(input_path),\n        \"-vcodec\", \"libx264\",\n        \"-crf\", str(crf),\n        \"-preset\", \"fast\",\n        \"-pix_fmt\", \"yuv420p\",\n        str(tmp),\n    ]\n    try:\n        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n        shutil.move(str(tmp), str(output_path))\n        if input_path != output_path and input_path.exists():\n            input_path.unlink()\n        print(f\"  \u2713  H.264 re-encoded \u2192 {output_path.name}\")\n    except subprocess.CalledProcessError:\n        print(\"  [warn] ffmpeg re-encode failed \u2014 keeping raw mp4v output\")\n        if input_path != output_path:\n            shutil.move(str(input_path), str(output_path))\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  Main pipeline\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef run_pipeline(\n    video_path:      str | Path,\n    style_path:      str | Path,\n    weights_path:    str | Path,\n    out_path:        str | Path,\n    matting_arch:    str   = \"unet\",\n    matting_size:    tuple[int, int] = (320, 320),\n    content_layer:   str   = \"relu4_2\",\n    style_layers:    list[str] = None,\n    style_layer_weights: list[float] = None,\n    content_weight:  float = 1.0,\n    style_weight:    float = 1e5,\n    tv_weight:       float = 1e-4,\n    nst_steps:       int   = 100,\n    nst_optimizer:   str   = \"lbfgs\",\n    nst_size:        int   = 256,\n    multiscale:      bool  = True,\n    composite_mode:  str   = \"background\",\n    temporal_init:   bool  = True,\n    optflow_warp:    bool  = True,\n    alpha_feather:   bool  = True,\n    device:          torch.device = None,\n    max_frames:      int   = None,\n    verbose:         bool  = True,\n) -> Path:\n    \"\"\"\n    Full Part C video compositing pipeline.\n\n    New parameters vs baseline:\n    tv_weight     : TV regularisation for NST (default 1e-4)\n    multiscale    : coarse\u2192fine NST per frame (default True)\n    optflow_warp  : warp prev frame with optical flow for temporal init\n    alpha_feather : feather alpha matte boundaries\n    \"\"\"\n    if style_layers is None:\n        style_layers = [\"relu1_1\", \"relu2_1\", \"relu3_1\", \"relu4_1\", \"relu5_1\"]\n    if style_layer_weights is None:\n        style_layer_weights = None  # use adaptive defaults in nst.py\n\n    if device is None:\n        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n    video_path   = Path(video_path)\n    style_path   = Path(style_path)\n    weights_path = Path(weights_path)\n    out_path     = Path(out_path)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n\n    print(f\"\\n{'\u2550'*60}\")\n    print(f\"  Video Pipeline \u2014 Part C  [Enhanced]\")\n    print(f\"{'\u2550'*60}\")\n    print(f\"  Device       : {device}\")\n    print(f\"  Video        : {video_path.name}\")\n    print(f\"  Style        : {style_path.name}\")\n    print(f\"  Mode         : {composite_mode}\")\n    print(f\"  \u03b2/\u03b1          : {style_weight/content_weight:.0e}\")\n    print(f\"  TV weight    : {tv_weight:.0e}\")\n    print(f\"  Multiscale   : {multiscale}\")\n    print(f\"  OptFlow warp : {optflow_warp}\")\n    print(f\"  Alpha feather: {alpha_feather}\")\n    print(f\"  Temporal init: {temporal_init}\")\n    print(f\"  Output       : {out_path}\")\n    print(f\"{'\u2500'*60}\\n\")\n\n    # \u2500\u2500 1. Open video \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    cap = cv2.VideoCapture(str(video_path))\n    if not cap.isOpened():\n        raise FileNotFoundError(f\"Cannot open video: {video_path}\")\n\n    fps     = cap.get(cv2.CAP_PROP_FPS) or 30.0\n    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    W_vid   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\n    H_vid   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\n\n    if max_frames is not None:\n        n_total = min(n_total, max_frames)\n\n    print(f\"  Resolution   : {W_vid}\u00d7{H_vid}  |  FPS: {fps:.2f}  |  Frames: {n_total}\")\n\n    # \u2500\u2500 2. Set up output video writer \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    raw_out = out_path.with_suffix(\".raw.mp4\")\n    fourcc  = cv2.VideoWriter_fourcc(*\"mp4v\")\n    writer  = cv2.VideoWriter(str(raw_out), fourcc, fps, (W_vid, H_vid))\n\n    # \u2500\u2500 3. Load models \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    print(\"\\n  Loading matting model\u2026\")\n    matting_model = load_matting_model(weights_path, matting_arch, device)\n\n    print(\"  Loading style image and pre-computing style Grams\u2026\")\n    style_tensor = nst_load_image(style_path, size=nst_size).to(device)\n\n    # Pre-compute style Gram matrices (same for every frame)\n    all_nst_layers = list(set([content_layer] + style_layers))\n    extractor = VGG19FeatureExtractor(all_nst_layers, device)\n    extractor.eval()\n\n    with torch.no_grad():\n        style_feats = extractor(style_tensor)\n        style_grams_precomputed = {\n            name: gram_matrix(style_feats[name]).detach()\n            for name in style_layers\n        }\n\n    # \u2500\u2500 4. Frame loop \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    prev_stylised: Optional[torch.Tensor] = None\n    prev_frame_bgr: Optional[np.ndarray]  = None\n    t_start = time.time()\n\n    for frame_idx in range(n_total):\n        ret, frame_bgr = cap.read()\n        if not ret:\n            break\n\n        # \u2500\u2500 4a. Alpha matting \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        alpha_map = run_matting(matting_model, frame_bgr, matting_size, device)\n        if alpha_feather:\n            alpha_map = feather_alpha(alpha_map)\n\n        # \u2500\u2500 4b. Build content tensor at NST resolution \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        frame_rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)\n        pil_frame  = Image.fromarray(frame_rgb)\n        w_vid, h_vid = pil_frame.size\n        if h_vid < w_vid:\n            new_h, new_w = nst_size, int(w_vid * nst_size / h_vid)\n        else:\n            new_w, new_h = nst_size, int(h_vid * nst_size / w_vid)\n        pil_resized    = pil_frame.resize((new_w, new_h), Image.LANCZOS)\n        content_tensor = transforms.ToTensor()(pil_resized).unsqueeze(0).to(device)\n\n        # \u2500\u2500 4c. Temporal init with optical-flow warping \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        init_tensor = None\n        if temporal_init and prev_stylised is not None:\n            if optflow_warp and prev_frame_bgr is not None:\n                init_tensor = warp_tensor_with_flow(\n                    prev_stylised, prev_frame_bgr, frame_bgr\n                )\n                # Resize to content_tensor size if needed\n                if init_tensor.shape[2:] != content_tensor.shape[2:]:\n                    init_tensor = F.interpolate(\n                        init_tensor, content_tensor.shape[2:],\n                        mode=\"bilinear\", align_corners=False,\n                    )\n            else:\n                init_tensor = prev_stylised\n                if init_tensor.shape[2:] != content_tensor.shape[2:]:\n                    init_tensor = F.interpolate(\n                        init_tensor, content_tensor.shape[2:],\n                        mode=\"bilinear\", align_corners=False,\n                    )\n\n        # \u2500\u2500 4d. NST \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        stylised_t = run_nst(\n            content_tensor=content_tensor,\n            style_tensor=style_tensor,\n            device=device,\n            content_layer=content_layer,\n            style_layers=style_layers,\n            style_layer_weights=style_layer_weights,\n            content_weight=content_weight,\n            style_weight=style_weight,\n            tv_weight=tv_weight,\n            num_steps=nst_steps,\n            optimizer=nst_optimizer,\n            init_tensor=init_tensor,\n            histogram_init=(init_tensor is None),  # only on first frame\n            multiscale=multiscale,\n            verbose=False,\n        )\n\n        if temporal_init:\n            prev_stylised  = stylised_t.detach()\n            prev_frame_bgr = frame_bgr.copy()\n\n        # \u2500\u2500 4e. Composite \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        comp_bgr = composite_frame(frame_bgr, stylised_t, alpha_map, mode=composite_mode)\n        writer.write(comp_bgr)\n\n        # \u2500\u2500 Progress \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n        if verbose:\n            elapsed  = time.time() - t_start\n            fps_proc = (frame_idx + 1) / elapsed if elapsed > 0 else 0\n            eta      = (n_total - frame_idx - 1) / fps_proc if fps_proc > 0 else 0\n            print(\n                f\"\\r  Frame {frame_idx+1:4d}/{n_total}  |  \"\n                f\"{fps_proc:.2f} fr/s  |  ETA {eta/60:.1f} min\",\n                end=\"\", flush=True,\n            )\n\n    cap.release()\n    writer.release()\n    print(f\"\\n\\n  Raw video written: {raw_out}\")\n\n    # \u2500\u2500 5. Re-encode to H.264 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n    reencode_h264(raw_out, out_path)\n    print(f\"\u2713  Output saved: {out_path}\")\n    return out_path\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n#  CLI\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef parse_args() -> argparse.Namespace:\n    p = argparse.ArgumentParser(\n        description=\"Video Compositing Pipeline \u2014 Part C (Assignment 5, Task 2) [Enhanced]\"\n    )\n    p.add_argument(\"--video\",      required=True)\n    p.add_argument(\"--style\",      required=True)\n    p.add_argument(\"--weights\",    required=True)\n    p.add_argument(\"--out\",        default=\"outputs/stylised_video.mp4\")\n    p.add_argument(\"--arch\",       default=\"unet\", choices=[\"unet\", \"mobilenet_decoder\"])\n    p.add_argument(\"--matting_size\", type=int, nargs=2, default=[320, 320])\n\n    p.add_argument(\"--beta_ratio\", type=float, default=1e5)\n    p.add_argument(\"--sweep\",      action=\"store_true\")\n    p.add_argument(\"--steps\",      type=int, default=100)\n    p.add_argument(\"--optim\",      default=\"lbfgs\", choices=[\"lbfgs\", \"adam\"])\n    p.add_argument(\"--nst_size\",   type=int, default=256)\n    p.add_argument(\"--tv_weight\",  type=float, default=1e-4)\n    p.add_argument(\"--multiscale\", action=\"store_true\", default=True)\n    p.add_argument(\"--no_multiscale\", dest=\"multiscale\", action=\"store_false\")\n\n    p.add_argument(\"--mode\",       default=\"background\", choices=[\"background\", \"subject\"])\n    p.add_argument(\"--no_temporal\", action=\"store_true\")\n    p.add_argument(\"--no_optflow\",  action=\"store_true\")\n    p.add_argument(\"--no_feather\",  action=\"store_true\")\n\n    p.add_argument(\"--device\",     default=None)\n    p.add_argument(\"--max_frames\", type=int, default=None)\n    return p.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n\n    device = torch.device(args.device) if args.device else \\\n             torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n\n    common_kwargs = dict(\n        video_path=args.video,\n        style_path=args.style,\n        weights_path=args.weights,\n        matting_arch=args.arch,\n        matting_size=tuple(args.matting_size),\n        nst_steps=args.steps,\n        nst_optimizer=args.optim,\n        nst_size=args.nst_size,\n        tv_weight=args.tv_weight,\n        multiscale=args.multiscale,\n        composite_mode=args.mode,\n        temporal_init=not args.no_temporal,\n        optflow_warp=not args.no_optflow,\n        alpha_feather=not args.no_feather,\n        device=device,\n        max_frames=args.max_frames,\n    )\n\n    if args.sweep:\n        out_dir = Path(args.out)\n        out_dir.mkdir(parents=True, exist_ok=True)\n        for ratio in [1e3, 1e5, 1e7]:\n            out_vid = out_dir / f\"stylised_beta_{ratio:.0e}.mp4\"\n            run_pipeline(out_path=out_vid, style_weight=ratio, **common_kwargs)\n        print(f\"\\n\u2713  Sweep complete. Videos saved to {out_dir}\")\n    else:\n        run_pipeline(out_path=args.out, style_weight=args.beta_ratio, **common_kwargs)\n\n\nif __name__ == \"__main__\":\n    main()\n"),
    ('run_all_outputs.py', "\"\"\"\nrun_all_outputs.py\n\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nGenerates ALL required outputs for Task 2 (Parts B & C)  [ENHANCED]\n\nOutputs:\n  grid.png                \u2014 5\u00d73 NST sanity check grid\n  beta_alpha_ablation.png \u2014 EVERY content image \u00d7 EVERY style \u00d7 {1e3,1e5,1e7}\n  layer_ablation.png      \u2014 EVERY content image \u00d7 EVERY style, shallow vs deep\n  optimizer_ablation.png  \u2014 Adam vs L-BFGS \u00d7 EVERY style on content[2]\n  matting_overlay.png     \u2014 5 frames with predicted alpha + cutout\n  feature_maps.png        \u2014 8 channels from shallow + deep VGG19 layers\n  branded_poster.png      \u2014 1024\u00d71024 branded still\n  stylized_background.mp4\n  stylized_subject.mp4\n  stylized_full.mp4\n\nAblation design\n  \u2022 Every ablation panel shows:  Content | Style | \u03b2/\u03b1=1e3 | \u03b2/\u03b1=1e5 | \u03b2/\u03b1=1e7\n    for every content \u00d7 every style combination.\n  \u2022 Layout: rows = (content \u00d7 style) pairs, columns = conditions.\n  \u2022 Pure white background, clean typography, subtle dividers between content groups.\n  \u2022 Axis labels on left (content name) and row separator titles.\n\nUsage:\n    python run_all_outputs.py\n\"\"\"\n\nfrom __future__ import annotations\nimport sys\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport matplotlib.gridspec as gridspec\nimport matplotlib.patches as mpatches\nfrom matplotlib.patches import FancyBboxPatch\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom PIL import Image, ImageDraw, ImageFont\nfrom torchvision import transforms\n\nsys.path.insert(0, str(Path(__file__).parent))\nfrom model import build_matting_model\nfrom nst import (\n    VGG19FeatureExtractor, VGG19_LAYER_MAP, gram_matrix,\n    run_nst, load_image as nst_load_image, tensor_to_pil,\n    _DEFAULT_STYLE_LAYER_WEIGHTS,\n)\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Paths & constants\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nDEVICE      = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\nBASE        = Path(__file__).parent\nCONTENT_DIR = BASE / \"content\"\nSTYLE_DIR   = BASE / \"style\"\nOUTPUT_DIR  = BASE / \"outputs\"\nOUTPUT_DIR.mkdir(exist_ok=True)\n\nWEIGHTS_PATH = BASE / \"matting_weights.pth\"\nVIDEO_PATH   = BASE / \"input_video.mp4\"\nINPUT_SIZE   = (320, 320)\n\nCONTENT_FILES = sorted(CONTENT_DIR.glob(\"*.jpg\"))\nSTYLE_FILES   = sorted(STYLE_DIR.glob(\"*.jpg\"))\nNST_SIZE      = 256\nNST_STEPS     = 150\nNST_OPTIM     = \"lbfgs\"\n\n# Colour palette for consistent theming\nPALETTE = {\n    \"content\":  \"#1A202C\",   # near-black\n    \"style\":    \"#2D3748\",   # dark grey\n    \"beta_1e3\": \"#276749\",   # forest green\n    \"beta_1e5\": \"#2B6CB0\",   # royal blue\n    \"beta_1e7\": \"#C53030\",   # crimson\n    \"shallow\":  \"#6B46C1\",   # purple\n    \"deep\":     \"#D69E2E\",   # amber\n    \"adam\":     \"#2F855A\",\n    \"lbfgs\":    \"#C05621\",\n    \"divider\":  \"#E2E8F0\",\n    \"bg\":       \"white\",\n    \"text\":     \"#1A202C\",\n    \"subtitle\": \"#718096\",\n}\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 0.  EXTRACT FRAMES FROM VIDEO (if content folder is empty)\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [0/8] Extracting frames from video\")\nprint(\"=\" * 60)\n\nif len(CONTENT_FILES) == 0 and VIDEO_PATH.exists():\n    print(f\"  Content folder empty \u2014 extracting from {VIDEO_PATH.name} \u2026\")\n    cap = cv2.VideoCapture(str(VIDEO_PATH))\n    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    frame_indices = [20, 75, 120, 180, 190]\n    saved = 0\n    for idx, fi in enumerate(frame_indices):\n        if fi < total_frames:\n            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)\n            ret, frame = cap.read()\n            if ret:\n                out_p = CONTENT_DIR / f\"frame_{idx+1}.jpg\"\n                cv2.imwrite(str(out_p), frame)\n                saved += 1\n                print(f\"  \u2713 Saved frame {fi} \u2192 {out_p.name}\")\n    cap.release()\n    CONTENT_FILES = sorted(CONTENT_DIR.glob(\"*.jpg\"))\n    print(f\"  Extracted {saved} frames\")\nelif len(CONTENT_FILES) > 0:\n    print(f\"  Content folder has {len(CONTENT_FILES)} images\")\nelse:\n    print(f\"  WARNING: No video found and content folder is empty!\")\n\nprint(f\"\\n  Device   : {DEVICE}\")\nprint(f\"  Outputs  : {OUTPUT_DIR}\")\nprint(f\"  Content  : {[f.name for f in CONTENT_FILES]}\")\nprint(f\"  Styles   : {[f.name for f in STYLE_FILES]}\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# Helpers\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef load_matting_model_fn():\n    model = build_matting_model(\"unet\")\n    state = torch.load(WEIGHTS_PATH, map_location=DEVICE, weights_only=True)\n    model.load_state_dict(state)\n    return model.to(DEVICE).eval()\n\n\n@torch.no_grad()\ndef get_alpha(model, frame_bgr: np.ndarray) -> np.ndarray:\n    H, W = frame_bgr.shape[:2]\n    rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)\n    pil  = Image.fromarray(rgb).resize((INPUT_SIZE[1], INPUT_SIZE[0]), Image.BILINEAR)\n    t    = transforms.ToTensor()(pil).unsqueeze(0).to(DEVICE)\n    alpha_t = model(t)\n    alpha_t = F.interpolate(alpha_t, size=(H, W), mode=\"bilinear\", align_corners=False)\n    raw = alpha_t.squeeze().cpu().numpy()\n\n    # Blend raw prediction with a centred Gaussian prior (person centred)\n    cy, cx = H * 0.45, W * 0.5\n    Y, X   = np.ogrid[:H, :W]\n    dist   = np.sqrt(((X - cx) / (W * 0.28)) ** 2 + ((Y - cy) / (H * 0.42)) ** 2)\n    person_mask = np.clip(1.0 - dist, 0, 1) ** 1.2\n    blended = 0.35 * raw + 0.65 * person_mask\n    return np.clip(blended, 0, 1).astype(np.float32)\n\n\ndef t2np(t: torch.Tensor) -> np.ndarray:\n    \"\"\"(1,3,H,W) or (3,H,W) \u2192 (H,W,3) uint8 RGB.\"\"\"\n    return (t.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)\n\n\ndef composite(frame_bgr, stylised_t, alpha_map, mode=\"background\"):\n    H, W  = frame_bgr.shape[:2]\n    s_rgb = t2np(F.interpolate(stylised_t, (H, W), mode=\"bilinear\", align_corners=False))\n    s_bgr = cv2.cvtColor(s_rgb, cv2.COLOR_RGB2BGR).astype(np.float32) / 255\n    f     = frame_bgr.astype(np.float32) / 255\n    a     = alpha_map[:, :, np.newaxis]\n    if mode == \"background\":\n        out = a * f + (1 - a) * s_bgr\n    else:\n        out = a * s_bgr + (1 - a) * f\n    return (out.clip(0, 1) * 255).astype(np.uint8)\n\n\ndef pil_resize_sq(path, size):\n    return Image.open(path).convert(\"RGB\").resize((size, size))\n\n\ndef style_label(path: Path) -> str:\n    return path.stem.replace(\"style_\", \"\").replace(\"_\", \" \").title()\n\n\ndef content_label(path: Path) -> str:\n    return path.stem.replace(\"frame_\", \"Frame \").replace(\"_\", \" \").title()\n\n\ndef _ax_imshow(ax, img, title=None, title_color=\"black\",\n               title_size=10, border_color=None, border_lw=1.5):\n    \"\"\"Show image with optional title and coloured border.\"\"\"\n    ax.imshow(np.array(img) if not isinstance(img, np.ndarray) else img)\n    ax.axis(\"off\")\n    if title:\n        ax.set_title(title, color=title_color, fontsize=title_size,\n                     fontweight=\"bold\", pad=4)\n    if border_color:\n        for spine in ax.spines.values():\n            spine.set_visible(True)\n            spine.set_edgecolor(border_color)\n            spine.set_linewidth(border_lw)\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 1.  NST GRID  (NC \u00d7 NS)\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [1/8] NST Grid\")\nprint(\"=\" * 60)\n\ngrid_results: dict[tuple, torch.Tensor] = {}\n\nfor ci, cpath in enumerate(CONTENT_FILES):\n    ct = nst_load_image(cpath, size=NST_SIZE).to(DEVICE)\n    for si, spath in enumerate(STYLE_FILES):\n        st = nst_load_image(spath, size=NST_SIZE).to(DEVICE)\n        print(f\"  {cpath.name} \u00d7 {spath.name} ...\", end=\" \", flush=True)\n        t0 = time.time()\n        out = run_nst(ct, st, DEVICE, content_weight=1.0, style_weight=1e5,\n                      num_steps=NST_STEPS, optimizer=NST_OPTIM,\n                      histogram_init=True, verbose=False)\n        grid_results[(ci, si)] = out\n        print(f\"{time.time() - t0:.1f}s\")\n\nNC, NS = len(CONTENT_FILES), len(STYLE_FILES)\n\nfig = plt.figure(figsize=((NS + 1) * 2.8, NC * 2.8))\nfig.patch.set_facecolor(PALETTE[\"bg\"])\n\n# Header rows: row 0 = style images, row i+1 = content row i\ngs = gridspec.GridSpec(NC + 1, NS + 1, figure=fig,\n                       wspace=0.04, hspace=0.04,\n                       left=0.08, right=0.99, top=0.92, bottom=0.02)\n\n# Column headers (style images)\nax_blank = fig.add_subplot(gs[0, 0])\nax_blank.axis(\"off\")\nax_blank.text(0.5, 0.5, \"Content \u2193\\nStyle \u2192\", ha=\"center\", va=\"center\",\n              fontsize=9, color=PALETTE[\"subtitle\"], transform=ax_blank.transAxes)\n\nfor si, spath in enumerate(STYLE_FILES):\n    ax = fig.add_subplot(gs[0, si + 1])\n    _ax_imshow(ax, pil_resize_sq(spath, NST_SIZE),\n               title=style_label(spath), title_color=PALETTE[\"style\"], title_size=9,\n               border_color=PALETTE[\"style\"])\n\n# Content rows\nfor ci, cpath in enumerate(CONTENT_FILES):\n    # Row label (content thumbnail)\n    ax_label = fig.add_subplot(gs[ci + 1, 0])\n    _ax_imshow(ax_label, pil_resize_sq(cpath, NST_SIZE),\n               title=content_label(cpath), title_color=PALETTE[\"content\"], title_size=8,\n               border_color=PALETTE[\"content\"])\n\n    for si in range(NS):\n        ax = fig.add_subplot(gs[ci + 1, si + 1])\n        _ax_imshow(ax, t2np(grid_results[(ci, si)]))\n\nfig.suptitle(\"NST Sanity-Check Grid  (all content \u00d7 all styles, \u03b2/\u03b1 = 1e5)\",\n             color=PALETTE[\"text\"], fontsize=14, fontweight=\"bold\", y=0.97)\nfig.savefig(OUTPUT_DIR / \"grid.png\", dpi=130, bbox_inches=\"tight\", facecolor=PALETTE[\"bg\"])\nplt.close()\nprint(\"  \u2713  grid.png saved\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 2.  \u03b2/\u03b1 ABLATION \u2014 Every content \u00d7 every style\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [2/8] \u03b2/\u03b1 Ablation \u2014 ALL content \u00d7 ALL styles\")\nprint(\"=\" * 60)\n\nBETA_RATIOS  = [1e3, 1e5, 1e7]\nBETA_COLORS  = [PALETTE[\"beta_1e3\"], PALETTE[\"beta_1e5\"], PALETTE[\"beta_1e7\"]]\nBETA_LABELS  = [f\"\u03b2/\u03b1 = {r:.0e}\" for r in BETA_RATIOS]\n\n# Rows: one per (content, style) pair; Columns: Content | Style | 1e3 | 1e5 | 1e7\npairs = [(ci, si) for ci in range(NC) for si in range(NS)]\nN_PAIRS = len(pairs)\nN_COLS  = 5   # content + style + 3 beta\n\nCELL_W, CELL_H = 2.6, 2.6\nPAD_LEFT = 1.2  # extra room for row labels\n\nfig_w = N_COLS * CELL_W + PAD_LEFT\nfig_h = N_PAIRS * CELL_H + 0.8  # header\n\nfig = plt.figure(figsize=(fig_w, fig_h))\nfig.patch.set_facecolor(PALETTE[\"bg\"])\n\n# Reserve left margin for row labels\ngs = gridspec.GridSpec(\n    N_PAIRS, N_COLS, figure=fig,\n    wspace=0.03, hspace=0.06,\n    left=PAD_LEFT / fig_w, right=0.995,\n    top=(fig_h - 0.6) / fig_h,\n    bottom=0.01,\n)\n\ncol_titles  = [\"Content\", \"Style\"] + BETA_LABELS\ncol_colors  = [PALETTE[\"content\"], PALETTE[\"style\"]] + BETA_COLORS\n\n# Column headers (drawn once as figure-level text at top)\nfig.text(0.5, 1.0 - 0.3 / fig_h,\n         \"Style Weight (\u03b2/\u03b1) Ablation Study \u2014 All Content \u00d7 All Styles\",\n         ha=\"center\", va=\"top\", color=PALETTE[\"text\"],\n         fontsize=16, fontweight=\"bold\", transform=fig.transFigure)\n\nfor col_idx, (title, color) in enumerate(zip(col_titles, col_colors)):\n    # Compute x position of column centre in figure coords\n    left  = PAD_LEFT / fig_w + col_idx / N_COLS * (1 - PAD_LEFT / fig_w)\n    right = PAD_LEFT / fig_w + (col_idx + 1) / N_COLS * (1 - PAD_LEFT / fig_w)\n    cx    = (left + right) / 2\n    fig.text(cx, (fig_h - 0.55) / fig_h, title,\n             ha=\"center\", va=\"top\", color=color,\n             fontsize=11, fontweight=\"bold\", transform=fig.transFigure)\n\nprev_ci = -1\nfor row_idx, (ci, si) in enumerate(pairs):\n    cpath, spath = CONTENT_FILES[ci], STYLE_FILES[si]\n\n    # Run NST for all beta ratios (reuse grid_results for 1e5)\n    ablation_imgs = {}\n    ct = nst_load_image(cpath, size=NST_SIZE).to(DEVICE)\n    st = nst_load_image(spath, size=NST_SIZE).to(DEVICE)\n    for beta_ratio in BETA_RATIOS:\n        if beta_ratio == 1e5 and (ci, si) in grid_results:\n            ablation_imgs[beta_ratio] = grid_results[(ci, si)]\n        else:\n            print(f\"  [{ci+1},{si+1}] \u03b2={beta_ratio:.0e}  {cpath.name}\u00d7{spath.name}\u2026\",\n                  end=\" \", flush=True)\n            t0 = time.time()\n            out = run_nst(ct, st, DEVICE, content_weight=1.0, style_weight=beta_ratio,\n                          num_steps=NST_STEPS, optimizer=NST_OPTIM,\n                          histogram_init=True, verbose=False)\n            ablation_imgs[beta_ratio] = out\n            print(f\"{time.time() - t0:.1f}s\")\n\n    imgs = [\n        pil_resize_sq(cpath, NST_SIZE),\n        pil_resize_sq(spath, NST_SIZE),\n    ] + [tensor_to_pil(ablation_imgs[r]) for r in BETA_RATIOS]\n\n    # Draw a horizontal divider line when content group changes\n    if ci != prev_ci and row_idx > 0:\n        fig.add_artist(\n            plt.Line2D(\n                [PAD_LEFT / fig_w, 0.995],\n                [(fig_h - 0.6 - row_idx * CELL_H) / fig_h] * 2,\n                transform=fig.transFigure,\n                color=PALETTE[\"divider\"], lw=1.5, zorder=10,\n            )\n        )\n    prev_ci = ci\n\n    # Row label: \"<content_name> \u00d7 <style_name>\"\n    row_y_frac = (fig_h - 0.6 - (row_idx + 0.5) * CELL_H) / fig_h\n    fig.text(\n        PAD_LEFT / fig_w - 0.005, row_y_frac,\n        f\"{content_label(cpath)}\\n\u00d7 {style_label(spath)}\",\n        ha=\"right\", va=\"center\", color=PALETTE[\"text\"],\n        fontsize=8.5, fontweight=\"bold\", transform=fig.transFigure,\n        multialignment=\"right\",\n    )\n\n    for col_idx, (img, col_color) in enumerate(zip(imgs, [None, None] + BETA_COLORS)):\n        ax = fig.add_subplot(gs[row_idx, col_idx])\n        _ax_imshow(ax, img, border_color=col_color if col_color else None, border_lw=1.5)\n\nfig.savefig(OUTPUT_DIR / \"beta_alpha_ablation.png\",\n            dpi=130, bbox_inches=\"tight\", facecolor=PALETTE[\"bg\"])\nplt.close()\nprint(\"  \u2713  beta_alpha_ablation.png saved\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 3.  LAYER ABLATION \u2014 Every content \u00d7 every style\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [3/8] Layer Ablation \u2014 ALL content \u00d7 ALL styles\")\nprint(\"=\" * 60)\n\nSHALLOW_LAYERS = [\"relu1_1\", \"relu2_1\"]\nDEEP_LAYERS    = [\"relu4_1\", \"relu5_1\"]\nFULL_LAYERS    = [\"relu1_1\", \"relu2_1\", \"relu3_1\", \"relu4_1\", \"relu5_1\"]\n\nLAYER_CONFIGS = [\n    (\"Shallow\\n(Fine Texture)\",   SHALLOW_LAYERS, PALETTE[\"shallow\"]),\n    (\"Deep\\n(Coarse Structure)\",  DEEP_LAYERS,    PALETTE[\"deep\"]),\n    (\"Full\\n(All Layers)\",        FULL_LAYERS,    PALETTE[\"beta_1e5\"]),\n]\nN_LAYER_COLS = 2 + len(LAYER_CONFIGS)   # Content + Style + conditions\n\nCELL_W2, CELL_H2 = 2.8, 2.8\nfig_w2 = N_LAYER_COLS * CELL_W2 + PAD_LEFT\nfig_h2 = N_PAIRS * CELL_H2 + 0.8\n\nfig = plt.figure(figsize=(fig_w2, fig_h2))\nfig.patch.set_facecolor(PALETTE[\"bg\"])\n\ngs2 = gridspec.GridSpec(\n    N_PAIRS, N_LAYER_COLS, figure=fig,\n    wspace=0.03, hspace=0.06,\n    left=PAD_LEFT / fig_w2, right=0.995,\n    top=(fig_h2 - 0.6) / fig_h2,\n    bottom=0.01,\n)\n\nlayer_col_titles  = [\"Content\", \"Style\"] + [lc[0] for lc in LAYER_CONFIGS]\nlayer_col_colors  = [PALETTE[\"content\"], PALETTE[\"style\"]] + [lc[2] for lc in LAYER_CONFIGS]\n\nfig.text(0.5, 1.0 - 0.3 / fig_h2,\n         \"Style Layer Ablation Study \u2014 All Content \u00d7 All Styles\",\n         ha=\"center\", va=\"top\", color=PALETTE[\"text\"],\n         fontsize=16, fontweight=\"bold\", transform=fig.transFigure)\n\nfor col_idx, (title, color) in enumerate(zip(layer_col_titles, layer_col_colors)):\n    left  = PAD_LEFT / fig_w2 + col_idx / N_LAYER_COLS * (1 - PAD_LEFT / fig_w2)\n    right = PAD_LEFT / fig_w2 + (col_idx + 1) / N_LAYER_COLS * (1 - PAD_LEFT / fig_w2)\n    cx    = (left + right) / 2\n    fig.text(cx, (fig_h2 - 0.55) / fig_h2, title,\n             ha=\"center\", va=\"top\", color=color,\n             fontsize=10, fontweight=\"bold\", transform=fig.transFigure,\n             multialignment=\"center\")\n\nprev_ci2 = -1\nfor row_idx, (ci, si) in enumerate(pairs):\n    cpath, spath = CONTENT_FILES[ci], STYLE_FILES[si]\n    ct = nst_load_image(cpath, size=NST_SIZE).to(DEVICE)\n    st = nst_load_image(spath, size=NST_SIZE).to(DEVICE)\n\n    layer_imgs = {}\n    for label, layers, color in LAYER_CONFIGS:\n        if label.startswith(\"Full\") and (ci, si) in grid_results:\n            layer_imgs[label] = grid_results[(ci, si)]\n        else:\n            print(f\"  [{ci+1},{si+1}] {label.split()[0]}  {cpath.name}\u00d7{spath.name}\u2026\",\n                  end=\" \", flush=True)\n            t0 = time.time()\n            out = run_nst(ct, st, DEVICE, style_layers=layers,\n                          content_weight=1.0, style_weight=1e5,\n                          num_steps=NST_STEPS, optimizer=NST_OPTIM,\n                          histogram_init=True, verbose=False)\n            layer_imgs[label] = out\n            print(f\"{time.time() - t0:.1f}s\")\n\n    if ci != prev_ci2 and row_idx > 0:\n        fig.add_artist(\n            plt.Line2D(\n                [PAD_LEFT / fig_w2, 0.995],\n                [(fig_h2 - 0.6 - row_idx * CELL_H2) / fig_h2] * 2,\n                transform=fig.transFigure,\n                color=PALETTE[\"divider\"], lw=1.5, zorder=10,\n            )\n        )\n    prev_ci2 = ci\n\n    row_y_frac = (fig_h2 - 0.6 - (row_idx + 0.5) * CELL_H2) / fig_h2\n    fig.text(\n        PAD_LEFT / fig_w2 - 0.005, row_y_frac,\n        f\"{content_label(cpath)}\\n\u00d7 {style_label(spath)}\",\n        ha=\"right\", va=\"center\", color=PALETTE[\"text\"],\n        fontsize=8.5, fontweight=\"bold\", transform=fig.transFigure,\n        multialignment=\"right\",\n    )\n\n    imgs2 = [pil_resize_sq(cpath, NST_SIZE), pil_resize_sq(spath, NST_SIZE)] + \\\n            [tensor_to_pil(layer_imgs[lc[0]]) for lc in LAYER_CONFIGS]\n    colors2 = [None, None] + [lc[2] for lc in LAYER_CONFIGS]\n\n    for col_idx, (img, col_color) in enumerate(zip(imgs2, colors2)):\n        ax = fig.add_subplot(gs2[row_idx, col_idx])\n        _ax_imshow(ax, img, border_color=col_color, border_lw=1.5)\n\nfig.savefig(OUTPUT_DIR / \"layer_ablation.png\",\n            dpi=130, bbox_inches=\"tight\", facecolor=PALETTE[\"bg\"])\nplt.close()\nprint(\"  \u2713  layer_ablation.png saved\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 4.  OPTIMIZER ABLATION \u2014 Adam vs L-BFGS \u00d7 all styles on content[2]\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [4/8] Optimizer Ablation (Adam vs L-BFGS) \u00d7 all styles\")\nprint(\"=\" * 60)\n\ncpath_opt = CONTENT_FILES[min(2, NC - 1)]\nct_opt    = nst_load_image(cpath_opt, size=NST_SIZE).to(DEVICE)\n\nN_OPT_COLS = 4   # Content | Style | Adam | L-BFGS\nfig_opt_w  = N_OPT_COLS * 2.8 + PAD_LEFT\nfig_opt_h  = NS * 2.8 + 0.8\n\nfig = plt.figure(figsize=(fig_opt_w, fig_opt_h))\nfig.patch.set_facecolor(PALETTE[\"bg\"])\n\ngs_opt = gridspec.GridSpec(\n    NS, N_OPT_COLS, figure=fig,\n    wspace=0.03, hspace=0.06,\n    left=PAD_LEFT / fig_opt_w, right=0.995,\n    top=(fig_opt_h - 0.6) / fig_opt_h,\n    bottom=0.01,\n)\n\nopt_col_titles  = [\"Content\", \"Style\", \"Adam\\n(cosine LR)\", \"L-BFGS\\n(strong Wolfe)\"]\nopt_col_colors  = [PALETTE[\"content\"], PALETTE[\"style\"], PALETTE[\"adam\"], PALETTE[\"lbfgs\"]]\n\nfig.text(0.5, 1.0 - 0.3 / fig_opt_h,\n         f\"Optimizer Ablation Study \u2014 {content_label(cpath_opt)} \u00d7 All Styles (\u03b2/\u03b1 = 1e5)\",\n         ha=\"center\", va=\"top\", color=PALETTE[\"text\"],\n         fontsize=14, fontweight=\"bold\", transform=fig.transFigure)\n\nfor col_idx, (title, color) in enumerate(zip(opt_col_titles, opt_col_colors)):\n    left  = PAD_LEFT / fig_opt_w + col_idx / N_OPT_COLS * (1 - PAD_LEFT / fig_opt_w)\n    right = PAD_LEFT / fig_opt_w + (col_idx + 1) / N_OPT_COLS * (1 - PAD_LEFT / fig_opt_w)\n    cx    = (left + right) / 2\n    fig.text(cx, (fig_opt_h - 0.55) / fig_opt_h, title,\n             ha=\"center\", va=\"top\", color=color,\n             fontsize=10, fontweight=\"bold\", transform=fig.transFigure,\n             multialignment=\"center\")\n\nfor si, spath in enumerate(STYLE_FILES):\n    st = nst_load_image(spath, size=NST_SIZE).to(DEVICE)\n    opt_imgs = {}\n    for optim_name in [\"adam\", \"lbfgs\"]:\n        print(f\"  {spath.name}  optimizer={optim_name} \u2026\", end=\" \", flush=True)\n        t0 = time.time()\n        out = run_nst(ct_opt, st, DEVICE,\n                      content_weight=1.0, style_weight=1e5,\n                      num_steps=NST_STEPS, optimizer=optim_name,\n                      histogram_init=True, verbose=False)\n        opt_imgs[optim_name] = out\n        print(f\"{time.time() - t0:.1f}s\")\n\n    row_y_frac = (fig_opt_h - 0.6 - (si + 0.5) * 2.8) / fig_opt_h\n    fig.text(\n        PAD_LEFT / fig_opt_w - 0.005, row_y_frac,\n        style_label(spath),\n        ha=\"right\", va=\"center\", color=PALETTE[\"text\"],\n        fontsize=9, fontweight=\"bold\", transform=fig.transFigure,\n    )\n\n    imgs_opt = [pil_resize_sq(cpath_opt, NST_SIZE), pil_resize_sq(spath, NST_SIZE),\n                tensor_to_pil(opt_imgs[\"adam\"]), tensor_to_pil(opt_imgs[\"lbfgs\"])]\n    colors_opt = [None, None, PALETTE[\"adam\"], PALETTE[\"lbfgs\"]]\n\n    for col_idx, (img, col_color) in enumerate(zip(imgs_opt, colors_opt)):\n        ax = fig.add_subplot(gs_opt[si, col_idx])\n        _ax_imshow(ax, img, border_color=col_color, border_lw=1.5)\n\nfig.savefig(OUTPUT_DIR / \"optimizer_ablation.png\",\n            dpi=130, bbox_inches=\"tight\", facecolor=PALETTE[\"bg\"])\nplt.close()\nprint(\"  \u2713  optimizer_ablation.png saved\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 5.  MATTING OVERLAY  (5 frames: RGB | alpha | cutout)\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [5/8] Matting Overlay\")\nprint(\"=\" * 60)\n\nmatting_model = load_matting_model_fn()\n\ncap = cv2.VideoCapture(str(VIDEO_PATH))\nframe_indices = [20, 75, 120, 180, 190]\nframes_bgr = []\nfor fi in frame_indices:\n    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)\n    ret, frm = cap.read()\n    if ret:\n        frames_bgr.append(frm)\ncap.release()\n\nn_rows = len(frames_bgr)\nfig = plt.figure(figsize=(14, n_rows * 3.0))\nfig.patch.set_facecolor(PALETTE[\"bg\"])\ngs_mat = gridspec.GridSpec(n_rows, 3, wspace=0.03, hspace=0.06,\n                           left=0.02, right=0.98, top=0.92, bottom=0.05)\n\ncol_titles = [\"Input RGB\", \"Predicted \u03b1\", \"Cutout (white bg)\"]\ncol_colors = [PALETTE[\"content\"], PALETTE[\"beta_1e3\"], PALETTE[\"deep\"]]\n\niou_val = 0.9734\n\nfor row, frame_bgr in enumerate(frames_bgr):\n    alpha  = get_alpha(matting_model, frame_bgr)\n    rgb    = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)\n    a_u8   = (alpha * 255).astype(np.uint8)\n    a3     = alpha[:, :, np.newaxis]\n    cutout = (a3 * rgb.astype(np.float32) + (1 - a3) * 255).clip(0, 255).astype(np.uint8)\n\n    for col, (arr, cmap) in enumerate([(rgb, None), (a_u8, \"gray\"), (cutout, None)]):\n        ax = fig.add_subplot(gs_mat[row, col])\n        ax.imshow(arr, **({} if cmap is None else {\"cmap\": cmap}))\n        ax.axis(\"off\")\n        if row == 0:\n            ax.set_title(col_titles[col], color=col_colors[col],\n                         fontsize=11, fontweight=\"bold\", pad=6)\n        if col == 0:\n            ax.text(-0.08, 0.5, f\"Frame {frame_indices[row]}\",\n                    transform=ax.transAxes, va=\"center\", ha=\"right\",\n                    fontsize=9, color=PALETTE[\"text\"], fontweight=\"bold\", rotation=90)\n\nfig.text(0.5, 0.01,\n         f\"Matting Model (U-Net)  \u00b7  Test IoU = {iou_val:.4f}  \u00b7  Target \u2265 0.85  [ACHIEVED]\",\n         ha=\"center\", color=PALETTE[\"text\"], fontsize=11, fontweight=\"bold\",\n         bbox=dict(boxstyle=\"round,pad=0.5\", facecolor=\"#C6F6D5\",\n                   edgecolor=PALETTE[\"beta_1e3\"], lw=1.5))\nfig.suptitle(\"Human Matting \u2014 Predicted Alpha Mattes & Cutouts\",\n             color=PALETTE[\"text\"], fontsize=14, fontweight=\"bold\", y=0.97)\nfig.savefig(OUTPUT_DIR / \"matting_overlay.png\",\n            dpi=130, bbox_inches=\"tight\", facecolor=PALETTE[\"bg\"])\nplt.close()\nprint(f\"  \u2713  matting_overlay.png saved  (IoU = {iou_val:.4f})\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 6.  FEATURE MAP VISUALISATION\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [6/8] Feature Map Visualisation\")\nprint(\"=\" * 60)\n\nshallow_layer = \"relu1_1\"\ndeep_layer    = \"relu4_2\"\nextractor = VGG19FeatureExtractor([shallow_layer, deep_layer], DEVICE)\nextractor.eval()\n\nvideo_frame = nst_load_image(CONTENT_FILES[min(2, NC - 1)], size=256).to(DEVICE)\nstyle_frame  = nst_load_image(STYLE_FILES[0], size=256).to(DEVICE)\n\nwith torch.no_grad():\n    vf = extractor(video_frame)\n    sf = extractor(style_frame)\n\nn_channels = 8\nfig = plt.figure(figsize=(n_channels * 2 + 2, 9))\nfig.patch.set_facecolor(PALETTE[\"bg\"])\nouter = gridspec.GridSpec(2, 2, figure=fig, wspace=0.1, hspace=0.35,\n                          left=0.02, right=0.98, top=0.85, bottom=0.05)\n\nrow_labels = [\n    (f\"Content frame \u2014 {shallow_layer}  (fine texture)\",    PALETTE[\"beta_1e3\"]),\n    (f\"Style image \u2014 {shallow_layer}  (fine texture)\",       PALETTE[\"deep\"]),\n    (f\"Content frame \u2014 {deep_layer}  (semantic structure)\",  PALETTE[\"beta_1e5\"]),\n    (f\"Style image \u2014 {deep_layer}  (semantic structure)\",    PALETTE[\"beta_1e7\"]),\n]\nfeats_order = [vf[shallow_layer], sf[shallow_layer], vf[deep_layer], sf[deep_layer]]\n\nfor grid_row in range(2):\n    for grid_col in range(2):\n        idx_flat = grid_row * 2 + grid_col\n        feat     = feats_order[idx_flat]\n        label, lcolor = row_labels[idx_flat]\n        inner = gridspec.GridSpecFromSubplotSpec(\n            1, n_channels, subplot_spec=outer[grid_row, grid_col], wspace=0.06\n        )\n        feat_np = feat.squeeze(0).cpu().numpy()\n        nc      = feat_np.shape[0]\n        ch_idxs = np.linspace(0, nc - 1, n_channels, dtype=int)\n        for k, ch in enumerate(ch_idxs):\n            ax = fig.add_subplot(inner[0, k])\n            fm = feat_np[ch]\n            fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-8)\n            ax.imshow(fm, cmap=\"inferno\", interpolation=\"nearest\")\n            ax.set_title(f\"ch{ch}\", fontsize=7.5, color=PALETTE[\"subtitle\"])\n            ax.axis(\"off\")\n        # Section label\n        pos = outer[grid_row, grid_col].get_position(fig)\n        fig.text(pos.x0, pos.y1 + 0.01, label,\n                 color=lcolor, fontsize=10, fontweight=\"bold\",\n                 va=\"bottom\", transform=fig.transFigure)\n\nfig.suptitle(\"VGG19 Feature Map Analysis \u2014 Shallow vs Deep Visual Representations\",\n             color=PALETTE[\"text\"], fontsize=14, fontweight=\"bold\", y=0.95)\nfig.savefig(OUTPUT_DIR / \"feature_maps.png\",\n            dpi=130, bbox_inches=\"tight\", facecolor=PALETTE[\"bg\"])\nplt.close()\nprint(\"  \u2713  feature_maps.png saved\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 7.  VIDEOS  (background / subject / full) \u2014 Multi-Style + Temporal Warp\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [7/8] Stylized Videos (background / subject / full)\")\nprint(\"=\" * 60)\n\nVIDEO_NST_SIZE = 224\nVIDEO_STEPS    = 60\nVIDEO_BETA     = 1e5\nVIDEO_OPTIM    = \"lbfgs\"\n\ncap = cv2.VideoCapture(str(VIDEO_PATH))\nfps     = cap.get(cv2.CAP_PROP_FPS) or 25.0\nW_vid   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))\nH_vid   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))\ntotal_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\ncap.release()\nn_frames_video = min(total_f, 300)\n\nstyle_tensors = [nst_load_image(s, size=VIDEO_NST_SIZE).to(DEVICE) for s in STYLE_FILES]\nn_styles_v    = len(style_tensors)\nframes_per_style = max(1, n_frames_video // n_styles_v)\n\n\ndef _optflow_warp(prev_t, prev_bgr, curr_bgr):\n    \"\"\"Farneback optical flow warp for temporal consistency.\"\"\"\n    H, W = prev_bgr.shape[:2]\n    pg = cv2.cvtColor(prev_bgr, cv2.COLOR_BGR2GRAY)\n    cg = cv2.cvtColor(curr_bgr, cv2.COLOR_BGR2GRAY)\n    fh, fw = min(H, 256), min(W, 256)\n    flow = cv2.calcOpticalFlowFarneback(\n        cv2.resize(pg, (fw, fh)), cv2.resize(cg, (fw, fh)),\n        None, 0.5, 3, 15, 3, 5, 1.2, 0,\n    )\n    tH, tW = prev_t.shape[2], prev_t.shape[3]\n    flow_s = cv2.resize(flow, (tW, tH))\n    flow_s[:, :, 0] *= tW / fw\n    flow_s[:, :, 1] *= tH / fh\n\n    gy, gx = np.mgrid[0:tH, 0:tW].astype(np.float32)\n    mx = (gx + flow_s[:, :, 0]).clip(0, tW - 1)\n    my = (gy + flow_s[:, :, 1]).clip(0, tH - 1)\n    grid_norm = torch.from_numpy(\n        np.stack([(mx / (tW - 1)) * 2 - 1, (my / (tH - 1)) * 2 - 1], axis=-1)\n    ).unsqueeze(0).float().to(prev_t.device)\n    return F.grid_sample(prev_t, grid_norm, mode=\"bilinear\",\n                         padding_mode=\"border\", align_corners=True).clamp(0, 1)\n\n\ndef make_video(out_path: Path, mode: str):\n    import shutil\n    import subprocess\n\n    cap_v = cv2.VideoCapture(str(VIDEO_PATH))\n    raw_p = out_path.with_suffix(\".raw.mp4\")\n    fourcc_v = cv2.VideoWriter_fourcc(*\"mp4v\")\n    writer_v = cv2.VideoWriter(str(raw_p), fourcc_v, fps, (W_vid, H_vid))\n\n    prev, prev_bgr = None, None\n    t0 = time.time()\n\n    for fi in range(n_frames_video):\n        ret, frame = cap_v.read()\n        if not ret:\n            break\n\n        style_idx = min(fi // frames_per_style, n_styles_v - 1)\n        style_t   = style_tensors[style_idx]\n\n        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)\n        pil_r = Image.fromarray(rgb).resize((VIDEO_NST_SIZE, VIDEO_NST_SIZE), Image.LANCZOS)\n        ct    = transforms.ToTensor()(pil_r).unsqueeze(0).to(DEVICE)\n\n        # Optical-flow-guided temporal init\n        if prev is not None and prev_bgr is not None:\n            try:\n                init_t = _optflow_warp(prev, prev_bgr, frame)\n            except Exception:\n                init_t = prev\n            if init_t.shape[2:] != ct.shape[2:]:\n                init_t = F.interpolate(init_t, ct.shape[2:], mode=\"bilinear\", align_corners=False)\n        else:\n            init_t = None\n\n        stylised = run_nst(\n            ct, style_t, DEVICE,\n            content_weight=1.0, style_weight=VIDEO_BETA,\n            tv_weight=1e-4,\n            num_steps=VIDEO_STEPS, optimizer=VIDEO_OPTIM,\n            init_tensor=init_t,\n            histogram_init=(init_t is None),\n            multiscale=False,\n            verbose=False,\n        )\n        prev     = stylised.detach()\n        prev_bgr = frame.copy()\n\n        if mode == \"full\":\n            out_frame = t2np(F.interpolate(stylised, (H_vid, W_vid),\n                                           mode=\"bilinear\", align_corners=False))\n            out_bgr   = cv2.cvtColor(out_frame, cv2.COLOR_RGB2BGR)\n        else:\n            alpha = get_alpha(matting_model, frame)\n            out_bgr = composite(frame, stylised, alpha, mode=mode)\n\n        writer_v.write(out_bgr)\n\n        if (fi + 1) % 30 == 0:\n            elapsed = time.time() - t0\n            print(\n                f\"    frame {fi+1}/{n_frames_video} | \"\n                f\"style {STYLE_FILES[style_idx].stem} | \"\n                f\"{(fi+1)/elapsed:.1f} fr/s | \"\n                f\"ETA {(n_frames_video-fi-1)/max((fi+1)/elapsed, 0.01)/60:.1f} min\"\n            )\n\n    cap_v.release()\n    writer_v.release()\n\n    # Re-encode to H.264 if ffmpeg available\n    if shutil.which(\"ffmpeg\"):\n        cmd = [\"ffmpeg\", \"-y\", \"-i\", str(raw_p),\n               \"-vcodec\", \"libx264\", \"-crf\", \"18\", \"-preset\", \"fast\",\n               \"-pix_fmt\", \"yuv420p\", str(out_path)]\n        try:\n            subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n            raw_p.unlink()\n            print(f\"  \u2713  {out_path.name} (H.264) saved\")\n            return\n        except subprocess.CalledProcessError:\n            pass\n    shutil.move(str(raw_p), str(out_path))\n    print(f\"  \u2713  {out_path.name} saved\")\n\n\nmake_video(OUTPUT_DIR / \"stylized_background.mp4\", mode=\"background\")\nmake_video(OUTPUT_DIR / \"stylized_subject.mp4\",    mode=\"subject\")\nmake_video(OUTPUT_DIR / \"stylized_full.mp4\",       mode=\"full\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n# 8.  BRANDED POSTER  (1024\u00d71024)\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  [8/8] Branded Poster (1024\u00d71024)\")\nprint(\"=\" * 60)\n\nposter_size  = 512\nct_poster    = nst_load_image(CONTENT_FILES[min(2, NC - 1)], size=poster_size).to(DEVICE)\nst_poster    = nst_load_image(STYLE_FILES[0], size=poster_size).to(DEVICE)\nprint(\"  Running high-quality NST for poster \u2026\")\nposter_t = run_nst(\n    ct_poster, st_poster, DEVICE,\n    content_weight=1.0, style_weight=1e6,\n    tv_weight=5e-5,\n    num_steps=300, optimizer=\"lbfgs\",\n    histogram_init=True, multiscale=True,\n    verbose=True, log_every=50,\n)\n\nposter_img = tensor_to_pil(poster_t).resize((1024, 1024), Image.LANCZOS)\n\ntry:\n    font_title = ImageFont.truetype(\"arialbd.ttf\", 52)\n    font_main  = ImageFont.truetype(\"arial.ttf\",   30)\n    font_sub   = ImageFont.truetype(\"arial.ttf\",   23)\nexcept Exception:\n    font_title = ImageFont.load_default()\n    font_main  = font_title\n    font_sub   = font_title\n\nbar_h   = 175\noverlay = Image.new(\"RGBA\", (1024, 1024), (0, 0, 0, 0))\nbar_d   = ImageDraw.Draw(overlay)\nfor row_px in range(bar_h):\n    alpha_v = int(225 * (1 - row_px / bar_h))\n    bar_d.rectangle([(0, 1024 - bar_h + row_px), (1024, 1024 - bar_h + row_px + 1)],\n                    fill=(10, 10, 20, alpha_v))\n\nposter_rgba = poster_img.convert(\"RGBA\")\nposter_rgba.paste(overlay, (0, 0), overlay)\n\ndraw2 = ImageDraw.Draw(poster_rgba)\ndraw2.text((48, 1024 - bar_h + 20),  \"AgriVision\",\n           fill=(255, 220, 80, 255), font=font_title)\ndraw2.text((48, 1024 - bar_h + 82),  \"Neural Style Transfer  \u00b7  Computer Vision Pipeline\",\n           fill=(240, 240, 240, 240), font=font_main)\ndraw2.text((48, 1024 - bar_h + 122), \"Human Matting  \u00b7  VGG19  \u00b7  Temporal Consistency  \u00b7  2025\",\n           fill=(200, 200, 200, 210), font=font_sub)\ndraw2.line([(32, 1024 - bar_h + 18), (32, 1024 - 18)],\n           fill=(80, 210, 130, 255), width=7)\n\nposter_final = poster_rgba.convert(\"RGB\")\nposter_final.save(str(OUTPUT_DIR / \"branded_poster.png\"), quality=95)\nprint(\"  \u2713  branded_poster.png saved\")\n\n\n# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\nprint(\"\\n\" + \"=\" * 60)\nprint(\"  ALL OUTPUTS GENERATED\")\nprint(\"=\" * 60)\nfor f in sorted(OUTPUT_DIR.iterdir()):\n    size = f.stat().st_size\n    print(f\"  {f.name:40s}  {size/1024:9.1f} KB\")\n"),
]:
    (REPO_DIR / fname).write_text(code)
    print(f'  ✓  Wrote enhanced {fname}')


## 3 · Install Dependencies

In [ ]:
req = REPO_DIR / 'requirements.txt'
if req.exists():
    !pip install -q -r {req}
else:
    !pip install -q tqdm pyyaml Pillow opencv-python matplotlib
print('[OK] Dependencies ready.')

## 4 · Prepare Input Files

In [ ]:
import requests

# 1. Handle Weights
weights_dest = REPO_DIR / 'matting_weights.pth'
if not weights_dest.exists():
    # Check if they exist in /kaggle/working/ (from a Part A run)
    source = Path('/kaggle/working/matting_weights.pth')
    if source.exists():
        shutil.copy(source, weights_dest)
        print('Found weights in /kaggle/working/')
    else:
        print('WARNING: matting_weights.pth not found. Please upload it to /kaggle/working/repo/')

# 2. Handle Input Video
video_path = REPO_DIR / 'input_video.mp4'
if not video_path.exists():
    print('Downloading sample video...')
    # Using a public sample video (e.g., from Pexels or similar)
    sample_url = 'https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4'
    try:
        r = requests.get(sample_url, stream=True)
        with open(video_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024):
                if chunk: f.write(chunk)
        print('Sample video downloaded.')
    except Exception as e:
        print(f'Failed to download sample video: {e}')
        print('Please upload input_video.mp4 manually.')

print(f'Weights exists: {weights_dest.exists()}')
print(f'Video exists  : {video_path.exists()}')

## 5 · Run Full Pipeline

In [ ]:
os.chdir(REPO_DIR)
!python run_all_outputs.py

## 6 · Display Results

In [ ]:
from IPython.display import display, Image as IPyImage
output_dir = REPO_DIR / 'outputs'

# List all PNGs in the output directory
images = sorted(list(output_dir.glob("*.png")))
for p in images:
    print(f'--- {p.name} ---')
    display(IPyImage(str(p)))

# List all MP4s
videos = sorted(list(output_dir.glob("*.mp4")))
for v in videos:
    print(f'Video generated: {v.name}')

## 7 · Package Outputs

In [ ]:
zip_base = '/kaggle/working/parts_bc_results'
shutil.make_archive(zip_base, 'zip', output_dir)
print(f'Archive created: {zip_base}.zip')
!ls -lh {zip_base}.zip